## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [7]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")

In [8]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)

In [9]:
from langchain_core.messages import HumanMessage
response=model.invoke([HumanMessage(content="Hi , My name is Mohammad. Iam a frontend developer")])
print(response.content)
print("---------------------------------------------------")
response=model.invoke([HumanMessage(content="Do you know my name?")])
print(response.content)

Hello Mohammad! 👋 Great to meet a fellow frontend developer. How can I assist you today? Whether you have questions about HTML, CSS, JavaScript frameworks, UI/UX best practices, or anything else, just let me know!
---------------------------------------------------
I don’t have any information about your name. If you’d like to share it, feel free to let me know! Otherwise, I’m happy to help with whatever you need.


In [10]:
from langchain_core.messages import AIMessage
# one conversation We built by hand. 
# A session is when the history is saved under an ID, 
# so the app can find and continue it later.
model.invoke(
    [
        HumanMessage(content="My name is Mohammad. Iam a frontend developer"),
        AIMessage(content="Hello Mohammad! It's nice to meet you. \n\nAs a frontend developer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **Mohammad**, and you work as a **frontend developer**.', additional_kwargs={'reasoning_content': 'The user asks: "Hey What\'s my name and what do I do?" We have prior conversation: user said "My name is Mohammad. Iam a frontend developer". So we should answer: Your name is Mohammad, you are a frontend developer. Should we add any extra? Probably just answer.'}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 146, 'total_tokens': 233, 'completion_time': 0.181583748, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.005915478, 'prompt_tokens_details': None, 'queue_time': 0.211368354, 'total_time': 0.187499226}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a207c3-12be-7391-a386-22cf36f4c4e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [11]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

/Users/mohammad.aldibes/Documents/LangChain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
config_1={"configurable":{"session_id":"chat1"}}

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Mohammad and I am a frontend developer")],
    config=config_1
)

/var/folders/4d/q3cdgnnx1_z7vjryt97hsv7w0000gq/T/ipykernel_26852/3391542844.py:8: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


In [16]:
response.content

'Hi Mohammad! 👋 Nice to meet you.  \n\nAs a frontend developer, you’ve got a lot of exciting tools and challenges at your fingertips. How can I help you today? Whether you want to discuss:\n\n- Latest trends in JavaScript frameworks (React, Vue, Svelte, etc.)  \n- CSS tricks, layout techniques, or design systems  \n- Performance optimization and tooling (Webpack, Vite, ESBuild)  \n- Testing strategies (Jest, Cypress, Playwright)  \n- Career advice, portfolio tips, or anything else frontend‑related  \n\nJust let me know what you’re interested in or any specific question you have!'

In [17]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config_1,
)

AIMessage(content='Your name is Mohammad.', additional_kwargs={'reasoning_content': 'User asks again "What\'s my name?" We previously responded "Your name is Mohammad." Should respond consistently.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 246, 'total_tokens': 282, 'completion_time': 0.076750329, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.012581959, 'prompt_tokens_details': None, 'queue_time': 0.047755292, 'total_time': 0.089332288}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a207c3-bd63-7566-9734-80aead50b606-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 246, 'output_tokens': 36, 'total_tokens': 282, 'output_token_details': {'reasoning': 22}})

In [ ]:
## change the config--> means to change session id
config_2={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config_2
)
response.content

'I don’t actually have any information about your name. If you’d like to share it, feel free to let me know!'

In [30]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John 🫰🫰🫰")],
    config=config_2
)
response.content

'Got it—your name is John. 😊 Anything specific you’d like to talk about or need help with?'

In [38]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name ???")],
    config=config_2
)
response.content

'Your name is John. Let me know if there’s anything else you’d like to discuss!'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.